# Cavity Resonance Phase Response & Pressure-Dependent Phase Sensitivity

Interactive exploration of the series RLC equivalent circuit model for an overmoded cavity.

**Key equations from the derivation:**

$$S_{21}(f) = \frac{A}{1 + j\,2Q_L\,\delta}$$

where $\delta = (f - f_0)/f_0$ is the fractional detuning, $A = 2Z_0/(2Z_0 + R_s)$ is the coupling amplitude, and $Q_L$ is the loaded quality factor.

$$\angle S_{21} = -\arctan(2Q_L \delta)$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
%matplotlib widget

## 1. Single-Mode $S_{21}$: Magnitude and Phase

Explore how the loaded Q and coupling amplitude shape the Lorentzian resonance.

- **$Q_L$** controls the width: higher Q = narrower resonance, steeper phase.
- **$R_s/Z_0$** controls the coupling amplitude $A = 1/(1 + R_s/2Z_0)$: higher loss = weaker transmission at resonance.

In [ ]:
def single_mode_s21(f, f0, Q_L, A):
    """
    Compute S21 for a single resonant cavity mode using the Lorentzian model.
    S21(f) = A / (1 + j * 2 * Q_L * delta), where delta = (f - f0) / f0
    
    Parameters:
        f    : frequency array (Hz)
        f0   : resonant frequency of the mode (Hz)
        Q_L  : loaded quality factor (accounts for internal loss + external port loading)
        A    : coupling amplitude at resonance = 2*Z0 / (2*Z0 + Rs)
    Returns:
        complex S21 array
    """
    delta = (f - f0) / f0  # fractional detuning from resonance
    return A / (1 + 1j * 2 * Q_L * delta)

# --- Set up dual-panel figure: magnitude (top) and phase (bottom) ---
fig1, (ax1a, ax1b) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig1.subplots_adjust(hspace=0.08)

f0_nominal = 60e9  # center frequency: 60 GHz (V-band)
f = np.linspace(f0_nominal * 0.995, f0_nominal * 1.005, 2000)  # +/- 0.5% around f0
f_GHz = f / 1e9

# Empty line objects that get updated by the slider callback
line_mag, = ax1a.plot([], [], 'b-', lw=1.5)
line_phase, = ax1b.plot([], [], 'r-', lw=1.5)

# Axis labels and formatting
ax1a.set_ylabel('$|S_{21}|$ (dB)')
ax1b.set_ylabel('$\\angle S_{21}$ (deg)')
ax1b.set_xlabel('Frequency (GHz)')
ax1a.set_title('Single Cavity Mode Response')
ax1a.grid(True, alpha=0.3)
ax1b.grid(True, alpha=0.3)
ax1a.set_xlim(f_GHz[0], f_GHz[-1])
ax1a.set_ylim(-40, 1)
ax1b.set_ylim(-95, 95)
ax1b.axhline(0, color='k', lw=0.5, ls='--')

# Text box showing computed parameters (coupling amplitude, bandwidth, phase slope)
info_text = ax1a.text(0.02, 0.05, '', transform=ax1a.transAxes, fontsize=9,
                      verticalalignment='bottom', fontfamily='monospace',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

def update_single_mode(Q_L=500, Rs_over_Z0=0.1):
    """Slider callback: recompute and redraw S21 for new Q_L and Rs/Z0."""
    # Coupling amplitude: A = 1 / (1 + Rs/(2*Z0))
    # A -> 1 for low loss (Rs << Z0), A -> 0 for high loss (Rs >> Z0)
    A = 1.0 / (1.0 + Rs_over_Z0 / 2.0)
    
    s21 = single_mode_s21(f, f0_nominal, Q_L, A)
    mag_dB = 20 * np.log10(np.abs(s21))
    phase_deg = np.degrees(np.angle(s21))
    
    # Update plot data
    line_mag.set_data(f_GHz, mag_dB)
    line_phase.set_data(f_GHz, phase_deg)
    
    # Derived quantities for the info box
    bw = f0_nominal / Q_L   # 3dB bandwidth (Hz)
    slope = -2 * Q_L        # d(angle S21)/d(delta) at resonance
    info_text.set_text(f'A = {A:.3f} ({20*np.log10(A):.1f} dB)\n'
                       f'BW = {bw/1e6:.1f} MHz\n'
                       f'd(phase)/d$\\delta$ = {slope:.0f}')
    fig1.canvas.draw_idle()

# Interactive sliders: Q_L on log scale (10 to 100k), Rs/Z0 on linear scale
interact(update_single_mode,
         Q_L=FloatLogSlider(value=500, base=10, min=1, max=5, step=0.1,
                            description='$Q_L$'),
         Rs_over_Z0=FloatSlider(value=0.1, min=0.01, max=10, step=0.01,
                                description='$R_s/Z_0$'));

## 2. Phase Slope at Resonance

The slope of the phase at resonance is $d(\angle S_{21})/d\delta\big|_{\delta=0} = -2Q_L$.

The entire 180 deg phase transition is compressed into a fractional bandwidth of ~$1/Q_L$.

In [ ]:
# Static comparison of phase response steepness for different Q_L values.
# Phase = -arctan(2 * Q_L * delta), so higher Q compresses the 180-deg
# transition into a narrower fractional bandwidth (~1/Q_L).

fig2, ax2 = plt.subplots(figsize=(9, 4))

delta = np.linspace(-0.02, 0.02, 2000)  # fractional detuning range: +/- 2%
for Q in [50, 200, 1000, 5000]:
    phase = -np.degrees(np.arctan(2 * Q * delta))  # angle(S21) = -arctan(2*Q_L*delta)
    ax2.plot(delta * 100, phase, lw=1.5, label=f'$Q_L$ = {Q}')

ax2.set_xlabel('Fractional detuning $\\delta$ (%)')
ax2.set_ylabel('$\\angle S_{21}$ (deg)')
ax2.set_title('Phase Steepness vs Loaded Q')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax2.axvline(0, color='k', lw=0.5, ls='--')
ax2.set_ylim(-95, 95)
fig2.tight_layout()

## 3. Multi-Mode Superposition

$$S_{21,\text{total}}(f) = \sum_{n=1}^{N} \frac{A_n}{1 + j\,2Q_{L,n}\,\delta_n}$$

In an overmoded cavity, many modes contribute to the total $S_{21}$. Modes with strong horn overlap (beam-like) carry the desired signal; modes with poor overlap add phase noise. Adjust the number of parasitic modes and their Q to see how the phase response degrades.

In [ ]:
# --- Multi-mode superposition: S21_total = sum of Lorentzian contributions ---
# Two classes of modes:
#   1. Beam-like modes: few, strong coupling (high A_n), moderate Q.
#      These have good spatial overlap with both Tx and Rx horns and
#      carry the desired through-transmission signal.
#   2. Parasitic modes: many, weak coupling (small A_n), potentially high Q.
#      Poor horn overlap, but collectively add phase noise to the total S21.

fig3, (ax3a, ax3b) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig3.subplots_adjust(hspace=0.08)

f_wide = np.linspace(58e9, 62e9, 5000)  # wider band: 58-62 GHz
f_wide_GHz = f_wide / 1e9

# Empty lines for interactive update
line3_mag, = ax3a.plot([], [], 'b-', lw=1)
line3_phase, = ax3b.plot([], [], 'r-', lw=1)
ax3a.set_ylabel('$|S_{21}|$ (dB)')
ax3b.set_ylabel('$\\angle S_{21}$ (deg)')
ax3b.set_xlabel('Frequency (GHz)')
ax3a.set_title('Multi-Mode Cavity Response')
ax3a.grid(True, alpha=0.3)
ax3b.grid(True, alpha=0.3)
ax3a.set_xlim(f_wide_GHz[0], f_wide_GHz[-1])
ax3a.set_ylim(-60, 5)
ax3b.set_ylim(-180, 180)

# Fixed random seed so parasitic mode positions are reproducible across slider moves
rng = np.random.default_rng(42)

def update_multimode(N_parasitic=20, Q_parasitic=2000, Q_beam=100, A_beam=0.9):
    """Slider callback: rebuild mode list and compute coherent phasor sum."""
    # Beam-like modes: 5 modes with hand-picked frequencies and relative amplitudes
    beam_freqs = np.array([59.0e9, 59.8e9, 60.0e9, 60.3e9, 61.0e9])
    beam_As = A_beam * np.array([0.5, 0.8, 1.0, 0.7, 0.4])  # scaled by slider
    beam_Qs = np.full(len(beam_freqs), Q_beam)

    # Parasitic modes: random frequencies and weak coupling amplitudes
    para_freqs = rng.uniform(58.5e9, 61.5e9, N_parasitic)
    para_As = rng.uniform(0.01, 0.15, N_parasitic)
    para_Qs = np.full(N_parasitic, Q_parasitic)

    # Combine both mode sets
    all_freqs = np.concatenate([beam_freqs, para_freqs])
    all_As = np.concatenate([beam_As, para_As])
    all_Qs = np.concatenate([beam_Qs, para_Qs])

    # Coherent phasor sum: S21_total(f) = sum_n [ A_n / (1 + j*2*Q_n*delta_n) ]
    s21_total = np.zeros_like(f_wide, dtype=complex)
    for fn, An, Qn in zip(all_freqs, all_As, all_Qs):
        delta_n = (f_wide - fn) / fn
        s21_total += An / (1 + 1j * 2 * Qn * delta_n)

    mag_dB = 20 * np.log10(np.abs(s21_total) + 1e-15)  # floor to avoid log(0)
    phase_deg = np.degrees(np.angle(s21_total))

    line3_mag.set_data(f_wide_GHz, mag_dB)
    line3_phase.set_data(f_wide_GHz, phase_deg)
    fig3.canvas.draw_idle()

interact(update_multimode,
         N_parasitic=IntSlider(value=20, min=0, max=200, step=5,
                               description='# parasitic'),
         Q_parasitic=FloatLogSlider(value=2000, base=10, min=1, max=5, step=0.1,
                                    description='$Q_{parasitic}$'),
         Q_beam=FloatLogSlider(value=100, base=10, min=0, max=4, step=0.1,
                               description='$Q_{beam}$'),
         A_beam=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05,
                            description='$A_{beam}$'));

## 4. Pressure-Dependent Phase Shift (Frequency Sweep)

With gas at pressure $p$, the refractive index $n(p) = 1 + \delta_n$ shifts every resonant frequency downward:

$$f_n(p) = \frac{f_n(0)}{n(p)}$$

The measured phase difference is:

$$\Delta\Phi(f) = \angle S_{21,\text{total}}(f,p) - \angle S_{21,\text{total}}(f,0)$$

The **direct path phase shift** (the desired signal) is:

$$\Delta\phi_{\text{direct}} = -\frac{2\pi f L_{\text{path}}}{c}\big(n(p) - 1\big)$$

High-Q cavity modes cause sign-flipping when their phase contribution $\Delta\phi_{\text{mode}}$ exceeds $\Delta\phi_{\text{direct}}$.

In [ ]:
# --- Pressure-dependent phase shift: frequency sweep at fixed pressure ---
# Three-panel plot decomposing the total measured phase difference into:
#   Top:    delta_phi_direct  — the desired refractive-index phase shift (smooth, ~linear in f)
#   Middle: delta_Phi_total   — what the MWI actually measures (direct + cavity modes)
#   Bottom: delta_phi_mode    — cavity-only contribution (erratic when modes have high Q)

fig4, (ax4a, ax4b, ax4c) = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
fig4.subplots_adjust(hspace=0.1)

c_light = 299792458.0  # speed of light (m/s)
f_sweep = np.linspace(59e9, 61e9, 5000)  # frequency sweep range: 59-61 GHz
f_sweep_GHz = f_sweep / 1e9

# Empty lines for each panel
line4_direct, = ax4a.plot([], [], 'g-', lw=1.5, label='$\\Delta\\phi_{direct}$ (theory)')
line4_total, = ax4b.plot([], [], 'b-', lw=1, label='$\\Delta\\Phi$ (total measured)')
line4_mode, = ax4c.plot([], [], 'm-', lw=1, label='$\\Delta\\phi_{mode}$ (cavity only)')

ax4a.set_ylabel('$\\Delta\\phi_{direct}$ (deg)')
ax4b.set_ylabel('$\\Delta\\Phi_{total}$ (deg)')
ax4c.set_ylabel('$\\Delta\\phi_{mode}$ (deg)')
ax4c.set_xlabel('Frequency (GHz)')
ax4a.set_title('Phase Difference: Pressure vs Vacuum (Frequency Sweep)')
for ax in [ax4a, ax4b, ax4c]:
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    ax.axhline(0, color='k', lw=0.5, ls='--')
ax4a.set_xlim(f_sweep_GHz[0], f_sweep_GHz[-1])

# Generate a random set of cavity modes (fixed seed for reproducibility)
rng4 = np.random.default_rng(7)
N_modes = 30
mode_freqs_vac = np.sort(rng4.uniform(58.5e9, 61.5e9, N_modes))  # vacuum resonant freqs
mode_As = rng4.uniform(0.02, 0.2, N_modes)  # coupling amplitudes

def compute_s21_total(f, freqs, As, Qs):
    """
    Coherent phasor sum over all cavity modes.
    S21_total(f) = sum_n [ A_n / (1 + j*2*Q_n * (f - f_n)/f_n) ]
    """
    s21 = np.zeros_like(f, dtype=complex)
    for fn, An, Qn in zip(freqs, As, Qs):
        delta_n = (f - fn) / fn
        s21 += An / (1 + 1j * 2 * Qn * delta_n)
    return s21

def update_pressure_freq(Q_modes=1000, pressure_Pa=1000, L_path_m=0.3):
    """Slider callback: compute phase difference between pressure p and vacuum."""
    # Refractive index shift: n(p) - 1 ≈ 2.9e-9 * P(Pa) for N2 at V-band
    dn = 2.9e-9 * pressure_Pa
    n_p = 1.0 + dn

    mode_Qs = np.full(N_modes, Q_modes)
    # Pressure shifts all resonant frequencies downward: f_n(p) = f_n(0) / n(p)
    mode_freqs_p = mode_freqs_vac / n_p

    # Direct path phase shift: dphi = -2*pi*f*L/c * (n-1)
    # This is the desired signal — smooth, proportional to pressure
    dphi_direct = -np.degrees(2 * np.pi * f_sweep * L_path_m / c_light * dn)

    # Compute cavity S21 at vacuum and at pressure, then take the phase difference
    s21_vac = compute_s21_total(f_sweep, mode_freqs_vac, mode_As, mode_Qs)
    s21_p = compute_s21_total(f_sweep, mode_freqs_p, mode_As, mode_Qs)

    # Cavity-induced phase change (can be erratic and cause sign flips)
    dphi_mode = np.degrees(np.angle(s21_p) - np.angle(s21_vac))
    # Total measured = direct path + cavity mode contributions
    dphi_total = dphi_direct + dphi_mode

    # Update all three panels
    line4_direct.set_data(f_sweep_GHz, dphi_direct)
    line4_total.set_data(f_sweep_GHz, dphi_total)
    line4_mode.set_data(f_sweep_GHz, dphi_mode)

    # Auto-scale y-axes
    ax4a.set_ylim(dphi_direct.min() * 1.2, max(dphi_direct.max() * 0.5, 0.1))
    pad = max(abs(dphi_total.max()), abs(dphi_total.min()), 1) * 1.3
    ax4b.set_ylim(-pad, pad)
    pad_m = max(abs(dphi_mode.max()), abs(dphi_mode.min()), 1) * 1.3
    ax4c.set_ylim(-pad_m, pad_m)
    fig4.canvas.draw_idle()

interact(update_pressure_freq,
         Q_modes=FloatLogSlider(value=1000, base=10, min=1, max=5, step=0.1,
                                description='$Q_{modes}$'),
         pressure_Pa=FloatLogSlider(value=1000, base=10, min=1, max=6, step=0.1,
                                    description='P (Pa)'),
         L_path_m=FloatSlider(value=0.3, min=0.05, max=1.0, step=0.01,
                              description='$L_{path}$ (m)'));

## 5. Pressure Sweep at Fixed Frequency

As pressure increases, all cavity resonances slide downward continuously. Each resonance that crosses the observation frequency swings the phase through a full arctan profile, temporarily dominating $\Delta\phi_{\text{direct}}$ and potentially reversing the sign of $\Delta\Phi$.

Compare the **ungridded** (high-Q) vs **gridded** (low-Q) configurations.

In [ ]:
# --- Pressure sweep at a fixed observation frequency ---
# As pressure rises, all cavity resonances slide downward (f_n(p) = f_n(0)/n(p)).
# Each resonance that crosses f_obs swings its phase contribution through the
# full arctan profile (+90 -> 0 -> -90 deg), potentially flipping the sign of
# the total measured phase.
#
# Top panel:  ungridded cavity (high Q) — expect erratic, non-monotonic behavior
# Bottom panel: gridded cavity (low Q)  — cavity modes suppressed, clean linear trend

fig5, axes5 = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig5.subplots_adjust(hspace=0.15)

pressures = np.linspace(0, 100000, 2000)  # 0 to 100 kPa

# Lines: blue = total measured, green dashed = ideal direct-path theory
line5_hi, = axes5[0].plot([], [], 'b-', lw=1, label='$\\Delta\\Phi$ (ungridded)')
line5_hi_d, = axes5[0].plot([], [], 'g--', lw=1.5, label='$\\Delta\\phi_{direct}$')
line5_lo, = axes5[1].plot([], [], 'r-', lw=1, label='$\\Delta\\Phi$ (gridded)')
line5_lo_d, = axes5[1].plot([], [], 'g--', lw=1.5, label='$\\Delta\\phi_{direct}$')

axes5[0].set_ylabel('Phase diff (deg)')
axes5[0].set_title('Pressure Sweep: Ungridded (high Q)')
axes5[1].set_ylabel('Phase diff (deg)')
axes5[1].set_title('Pressure Sweep: Gridded (low Q)')
axes5[1].set_xlabel('Pressure (kPa)')
for ax in axes5:
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    ax.axhline(0, color='k', lw=0.5, ls='--')

# Random cavity modes for this section (different seed than section 4)
rng5 = np.random.default_rng(99)
N5 = 40
mode_f0s = rng5.uniform(59e9, 61e9, N5)    # vacuum resonant frequencies
mode_A5s = rng5.uniform(0.02, 0.15, N5)    # coupling amplitudes

def update_pressure_sweep(f_obs_GHz=60.0, Q_ungridded=3000, Q_gridded=50, L_path_m=0.3):
    """Slider callback: sweep pressure from 0 to 100 kPa at fixed f_obs."""
    f_obs = f_obs_GHz * 1e9
    f_obs_arr = np.array([f_obs])  # single-element array for compute_s21_total

    Qs_hi = np.full(N5, Q_ungridded)  # ungridded: high Q for all modes
    Qs_lo = np.full(N5, Q_gridded)    # gridded: low Q (grid introduces radiation loss)

    # Vacuum reference: S21 at the observation frequency with no gas
    s21_vac_hi = compute_s21_total(f_obs_arr, mode_f0s, mode_A5s, Qs_hi)[0]
    s21_vac_lo = compute_s21_total(f_obs_arr, mode_f0s, mode_A5s, Qs_lo)[0]

    dphi_direct = np.zeros_like(pressures)
    dphi_total_hi = np.zeros_like(pressures)
    dphi_total_lo = np.zeros_like(pressures)

    for i, p in enumerate(pressures):
        # Refractive index at this pressure
        dn = 2.9e-9 * p
        n_p = 1.0 + dn

        # Direct-path phase shift (the desired measurement)
        dphi_direct[i] = -np.degrees(2 * np.pi * f_obs * L_path_m / c_light * dn)

        # Shift all mode frequencies downward by the refractive index
        shifted_freqs = mode_f0s / n_p

        # S21 at pressure for both ungridded and gridded
        s21_p_hi = compute_s21_total(f_obs_arr, shifted_freqs, mode_A5s, Qs_hi)[0]
        s21_p_lo = compute_s21_total(f_obs_arr, shifted_freqs, mode_A5s, Qs_lo)[0]

        # Total = direct path + cavity mode phase change
        dphi_total_hi[i] = dphi_direct[i] + np.degrees(np.angle(s21_p_hi) - np.angle(s21_vac_hi))
        dphi_total_lo[i] = dphi_direct[i] + np.degrees(np.angle(s21_p_lo) - np.angle(s21_vac_lo))

    # Update plot data
    p_kPa = pressures / 1000
    line5_hi.set_data(p_kPa, dphi_total_hi)
    line5_hi_d.set_data(p_kPa, dphi_direct)
    line5_lo.set_data(p_kPa, dphi_total_lo)
    line5_lo_d.set_data(p_kPa, dphi_direct)

    # Auto-scale y-axes
    for ax in axes5:
        ax.set_xlim(0, 100)
    pad_hi = max(np.abs(dphi_total_hi).max(), np.abs(dphi_direct).max(), 1) * 1.3
    axes5[0].set_ylim(-pad_hi, pad_hi)
    pad_lo = max(np.abs(dphi_total_lo).max(), np.abs(dphi_direct).max(), 1) * 1.3
    axes5[1].set_ylim(-pad_lo, pad_lo)
    fig5.canvas.draw_idle()

interact(update_pressure_sweep,
         f_obs_GHz=FloatSlider(value=60.0, min=59.0, max=61.0, step=0.01,
                               description='$f_{obs}$ (GHz)'),
         Q_ungridded=FloatLogSlider(value=3000, base=10, min=1, max=5, step=0.1,
                                    description='$Q_{ungridded}$'),
         Q_gridded=FloatLogSlider(value=50, base=10, min=0, max=4, step=0.1,
                                  description='$Q_{gridded}$'),
         L_path_m=FloatSlider(value=0.3, min=0.05, max=1.0, step=0.01,
                              description='$L_{path}$ (m)'));